# AWS Bedrock Prompt Management Demo

**Reference:** Slide 30 (*Hands-On RAG with AWS*)

This notebook demonstrates the lifecycle of **AWS Bedrock Prompt Management** using `boto3.client('bedrock-agent')` and invocation via the Bedrock Converse API (`boto3.client('bedrock-runtime')`).

### Key Concepts Covered:
1. **Prompt Creation (`create_prompt`)**: Defining model parameters, system prompts, variants, and dynamic `{{variables}}`.
2. **Immutable Versioning (`create_prompt_version`)**: Freezing versions for staging, testing, and production deployments.
3. **Variant Retrieval (`get_prompt`)**: Fetching versioned metadata and prompt schemas.
4. **Parameter Hydration & Validation**: Validating input variables and resolving placeholders.
5. **Converse API Invocation**: Passing hydrated prompts dynamically to Anthropic Claude or Amazon Nova models.
6. **Teardown & Cleanup**: Removing temporary prompt resources.

In [ ]:
import os
import sys
import time
import boto3
from prompt_management_demo import BedrockPromptManager

# Initialize BedrockPromptManager (auto-detects credentials or falls back to mock mode)
manager = BedrockPromptManager(region_name="us-east-1")
print(f"Manager initialized. Mock Mode: {manager.mock_mode}")

## 1. Create a Managed Prompt with Variants
We specify the prompt template with `{{context}}` and `{{query}}` placeholders, targeted model, and temperature.

In [ ]:
prompt_name = f"bedrock-rag-prompt-{int(time.time())}"
template = (
    "You are a specialized enterprise AI assistant.\n\n"
    "Retrieved Knowledge Base Context:\n{{context}}\n\n"
    "User Query: {{query}}\n\n"
    "Synthesize an accurate answer directly from the context. "
    "If the answer cannot be determined from the context, state that clearly."
)

prompt = manager.create_prompt(
    name=prompt_name,
    template_text=template,
    input_variables=["context", "query"],
    model_id="anthropic.claude-3-haiku-20240307-v1:0",
    description="Interactive demo prompt for enterprise RAG",
    temperature=0.2,
    max_tokens=1024
)
prompt_id = prompt.get("id")
print(f"Created Prompt ID: {prompt_id}")
print(f"Default Variant: {prompt.get('defaultVariant')}")

## 2. Publish an Immutable Version
Versions in Bedrock Prompt Management allow you to maintain immutable snapshots for production stability.

In [ ]:
version_response = manager.create_prompt_version(
    prompt_id=prompt_id,
    description="Production Release 1.0"
)
version_num = version_response.get("version", "1")
print(f"Published Version: {version_num}")

## 3. Retrieve Versioned Prompt
Fetch the frozen version and inspect its configuration.

In [ ]:
retrieved_prompt = manager.get_prompt(prompt_id=prompt_id, prompt_version=version_num)
print("Retrieved Prompt Details:")
print(f" - Name: {retrieved_prompt.get('name')}")
print(f" - Version: {retrieved_prompt.get('version')}")
print(f" - Model ID: {retrieved_prompt['variants'][0]['modelId']}")

## 4. Parameter Hydration & Converse API Invocation
Hydrate the dynamic variables and execute the call against Bedrock Converse API.

In [ ]:
payload = {
    "context": "Amazon Bedrock Knowledge Bases automatically orchestrates end-to-end RAG workflows including data ingestion, chunking, embedding generation, vector store persistence, and query-time retrieval.",
    "query": "What workflow steps does Bedrock Knowledge Bases orchestrate?"
}

execution_result = manager.execute_with_converse(
    prompt_id=prompt_id,
    variables=payload,
    prompt_version=version_num
)

print("\n--- Response Output ---")
print(execution_result["output"]["message"]["content"][0]["text"])
print(f"\nUsage: {execution_result.get('usage')}")

## 5. Teardown / Cleanup

In [ ]:
cleanup_success = manager.delete_prompt(prompt_id)
print(f"Prompt cleaned up: {cleanup_success}")